In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/olandr1ach4n/cyclistic-bikeshare-cleaned/cyclistic_cleaned.csv


Loading the data

In [2]:
bike_df = pd.read_csv(
    "/kaggle/input/datasets/olandr1ach4n/cyclistic-bikeshare-cleaned/cyclistic_cleaned.csv"
)

/tmp/ipykernel_16/2441204267.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  bike_df = pd.read_csv(


In [3]:
bike_df.shape

(6288920, 18)

In [4]:
bike_df["ride_id"].dtype

dtype('O')

In [5]:
bike_df["ride_id"].head()

0    E555D9F8E9C2F56B
1    EAEBE70DBF015886
2    A42F8404F804E56E
3    E49D01E38A1E074E
4    9308F609195D2414
Name: ride_id, dtype: object

In [6]:
bike_df["ride_id"].sample(20)

3864826       E330AE32872F1862
5570258    7478956147780187418
5522374    3346153742186411602
3982577       FD8E9F3083CA46E6
5898085     500906026818371038
4971339       8A05675AF581FD9C
1951526       733239A180DD5074
3943725       FB33EAA6FD5747AF
3164480       491EA0D0750E5ED9
5691510    8520059941512230734
4859480       16411D8DAB5870D1
1470240       2BB1568735C51277
4256549       C194A1546C49BFCA
2675745       26F8E8EED1744FAC
1180384       A96A669ED24EF8F5
5018210       717950E0812E905F
3740051       27B0C1B89995F3EA
2926304       6FAF7C6124317070
6222158    6104504922924841306
943186        8CF833E724403E6F
Name: ride_id, dtype: object

In [7]:
# force ride_id to be string when loading
bike_df = pd.read_csv(
    "/kaggle/input/datasets/olandr1ach4n/cyclistic-bikeshare-cleaned/cyclistic_cleaned.csv",
    dtype={"ride_id": str},
    low_memory=False
)

In [8]:
# verify no missing ride_ids
bike_df["ride_id"].isna().sum()

np.int64(0)

In [9]:
# verify if ride_ids are string
bike_df["ride_id"].apply(type).value_counts()

ride_id
<class 'str'>    6288920
Name: count, dtype: int64

In [10]:
bike_df.info()
bike_df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6288920 entries, 0 to 6288919
Data columns (total 18 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             object 
 1   rideable_type       object 
 2   started_at          object 
 3   ended_at            object 
 4   start_station_name  object 
 5   start_station_id    object 
 6   end_station_name    object 
 7   end_station_id      object 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       object 
 13  ride_length         float64
 14  date                object 
 15  month               object 
 16  day_of_week         object 
 17  hour                int64  
dtypes: float64(5), int64(1), object(12)
memory usage: 863.7+ MB


(6288920, 18)

# **Analysis Phase** 

In [11]:
# Member vs. Casual ride counts
# Who uses cyclistic more frequently?

bike_df["member_casual"].value_counts()

member_casual
member    4053818
casual    2235102
Name: count, dtype: int64

In [12]:
# Average Ride Length
# Who takes longer rides?
bike_df.groupby("member_casual")["ride_length"].mean()

member_casual
casual    17.399182
member    11.867092
Name: ride_length, dtype: float64

In [13]:
# Ride Count by Day of Week
# Do members commute while casual riders ride recreationally?
import pandas as pd

ride_day = pd.crosstab(
    bike_df["day_of_week"],
    bike_df["member_casual"]
)

ride_day

member_casual,casual,member
day_of_week,,
Friday,345110,592863
Monday,257973,566624
Saturday,481269,530976
Sunday,362184,430596
Thursday,271635,627131
Tuesday,259977,660840
Wednesday,256954,644788


In [14]:
# Average ride length of the day
bike_df.groupby(
    ["member_casual","day_of_week"]
)["ride_length"].mean()

member_casual  day_of_week
casual         Friday         17.044938
               Monday         17.519202
               Saturday       19.562827
               Sunday         19.960739
               Thursday       15.241759
               Tuesday        15.209443
               Wednesday      14.587610
member         Friday         11.778884
               Monday         11.567266
               Saturday       13.069188
               Sunday         12.940816
               Thursday       11.398668
               Tuesday        11.473854
               Wednesday      11.363343
Name: ride_length, dtype: float64

In [15]:
# Ride Count by Month
pd.crosstab(
    bike_df["month"],
    bike_df["member_casual"]
)

member_casual,casual,member
month,,
April,131211,316667
August,346450,520942
December,28004,112415
February,41083,160241
January,24654,112923
July,357085,511198
June,307847,454017
March,87615,229110
May,244628,408496


In [16]:
# Bike Type Preference
pd.crosstab(
    bike_df["rideable_type"],
    bike_df["member_casual"]
)

member_casual,casual,member
rideable_type,,
classic_bike,531795,1221962
electric_bike,1703307,2831856


In [17]:
# Peak riding hours
pd.crosstab(
    bike_df["hour"],
    bike_df["member_casual"]
)

member_casual,casual,member
hour,,
0,49755,39483
1,32900,24203
2,20536,14121
3,12008,9363
4,9342,10121
5,15063,39105
6,31209,114385
7,54916,224728
8,79794,293303


In [18]:
# calculating percentages
member_pct = (
    bike_df["member_casual"]
    .value_counts(normalize=True)
    * 100
)

member_pct

member_casual
member    64.459685
casual    35.540315
Name: proportion, dtype: float64

Export necessary csv files for visualization

In [19]:
os.makedirs("dashboard_data", exist_ok=True)

In [20]:
# kpi dataset
kpi_df = pd.DataFrame({
    "Metric": [
        "Total Rides",
        "Member %",
        "Casual %",
        "Avg Member Ride",
        "Avg Casual Ride"
    ],
    "Value": [
        len(bike_df),
        64.459685,
        35.540315,
        11.867092,
        17.399182
    ]
})

kpi_df.to_csv(
    "dashboard_data/kpi_summary.csv",
    index=False
)

kpi_df

,Metric,Value
0,Total Rides,6.288920e+06
1,Member %,6.445968e+01
2,Casual %,3.554031e+01
3,Avg Member Ride,1.186709e+01
4,Avg Casual Ride,1.739918e+01


In [21]:
# ride count by rider type 
ride_count_df = (
    bike_df["member_casual"]
    .value_counts()
    .reset_index()
)

ride_count_df.columns = [
    "member_casual",
    "ride_count"
]

ride_count_df.to_csv(
    "dashboard_data/ride_count_by_type.csv",
    index=False
)

In [22]:
# average ride length
avg_ride_df = (
    bike_df.groupby("member_casual")["ride_length"]
    .mean()
    .reset_index()
)

avg_ride_df.to_csv(
    "dashboard_data/avg_ride_length.csv",
    index=False
)

In [23]:
# day of week
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

In [24]:
weekday_df = pd.crosstab(
    bike_df["day_of_week"],
    bike_df["member_casual"]
)

weekday_df = weekday_df.reindex(day_order)

weekday_df.to_csv(
    "dashboard_data/weekday_counts.csv"
)

weekday_df.head()

member_casual,casual,member
day_of_week,,
Monday,257973,566624
Tuesday,259977,660840
Wednesday,256954,644788
Thursday,271635,627131
Friday,345110,592863


In [25]:
# monthly trend 
month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

In [26]:
month_df = pd.crosstab(
    bike_df["month"],
    bike_df["member_casual"]
)

month_df = month_df.reindex(month_order)

month_df.to_csv(
    "dashboard_data/monthly_counts.csv"
)

In [27]:
# hourly usage
hour_df = pd.crosstab(
    bike_df["hour"],
    bike_df["member_casual"]
)

hour_df.to_csv(
    "dashboard_data/hourly_counts.csv"
)

In [28]:
# bike preference
bike_type_df = pd.crosstab(
    bike_df["rideable_type"],
    bike_df["member_casual"]
)

bike_type_df.to_csv(
    "dashboard_data/bike_type_counts.csv"
)

In [29]:
import shutil

shutil.make_archive(
    "cyclistic_dashboard_data",
    "zip",
    "dashboard_data"
)


'/kaggle/working/cyclistic_dashboard_data.zip'